# 1 · Scene cataloguing and imagery download


This notebook reproduces the first two steps of the pipeline: finding cloud-free Sentinel-2 scenes over the Lagos–Lekki–Badagry study area and downloading the green, SWIR and scene-classification bands needed for shoreline extraction.

**Data source.** Sentinel-2 Level-2A (atmospherically corrected) imagery is distributed by the ESA Copernicus programme. The scene catalogue is queried through the [Copernicus Data Space Ecosystem STAC API](https://dataspace.copernicus.eu/), which is open and requires no account for searching. The raster bands are then read from the public Amazon S3 mirror of the Sentinel-2 archive ([registry.opendata.aws/sentinel-2-l2a](https://registry.opendata.aws/sentinel-2-l2a/)), also anonymous.

**Study area.** The bounding box `(2.9°E–4.35°E, 6.15°N–6.85°N)` covers the Atlantic shoreline from Badagry through Lagos, Victoria Island and Oniru to the Lekki–Epe stretch, crossing MGRS tiles `31NEH` and `31NFH`.

**Scene selection.** We keep the two clearest scenes per calendar year (scene-level cloud cover ≤ 15 %) between 2017 and 2025. Scenes are additionally verified against the archive before download, so the catalogue only contains data that actually exists on the mirror. Note that the public L2A mirror for this region only stores scenes from 2017 onward; the 2015–2016 part of the planned decade is therefore documented as a data-availability limitation in the README.


In [ ]:
import json
from pathlib import Path

import requests

BASE = Path("..")
STAC_URL = ("https://catalogue.dataspace.copernicus.eu/stac/search")
COLLECTION = "sentinel-2-l2a"
BBOX = [2.9, 6.15, 4.35, 6.85]

query = {
    "collections": [COLLECTION],
    "bbox": BBOX,
    "datetime": "2017-01-01T00:00:00Z/2025-12-31T23:59:59Z",
    "limit": 500,
    "query": {"eo:cloud_cover": {"lte": 15}},
}
response = requests.post(STAC_URL, json=query, timeout=60)
items = response.json()["features"]
print(f"{len(items)} scenes found over the study area")

# ---------------------------------------------------------------------------
# Keep the two clearest scenes per year (prefer winter, when the Harmattan
# haze is usually weakest on the Atlantic side of the coast)
# ---------------------------------------------------------------------------
by_year: dict[str, list] = {}
for item in items:
    props = item["properties"]
    year = props["datetime"][:4]
    if year not in by_year:
        by_year[year] = []
    by_year[year].append({
        "id": item["id"],
        "tile": props["s2:mgrs_tile"],
        "datetime": props["datetime"],
        "cloud_cover": props["eo:cloud_cover"],
    })

scenes = []
for year in sorted(by_year):
    candidates = sorted(by_year[year], key=lambda s: s["cloud_cover"])[:2]
    scenes.extend(candidates)
print(f"{len(scenes)} scenes kept: "
      f"{', '.join(s['id'][17:25] for s in scenes)}")

catalog = {"study_area_bbox": BBOX, "collection": COLLECTION,
           "generated_by": "notebook 01", "scenes": scenes}
(BASE / "data" / "scene_catalog.json").write_text(json.dumps(
    catalog, indent=2))
print("catalogue saved to data/scene_catalog.json")



In [ ]:
# ---------------------------------------------------------------------------
# Download the bands needed for MNDWI + cloud masking (B03 green, B04 red,
# B11 SWIR-1 at 20 m and the SCL scene-classification layer). The bands are
# fetched from the public S3 mirror with anonymous (unsigned) access.
# ---------------------------------------------------------------------------
import boto3
from botocore import UNSIGNED, config as botocore_config

from scripts.config import RAW_DIR

s3 = boto3.client("s3", config=botocore_config.Config(
    signature_version=UNSIGNED))

for scene in catalog["scenes"]:
    scene_dir = RAW_DIR / scene["tile"] / scene["datetime"][:10]
    scene_dir.mkdir(parents=True, exist_ok=True)
    if all((scene_dir / f"{band}.jp2").exists()
           for band in ("B03", "B04", "B11", "SCL")):
        print(f"{scene['id']}: already present")
        continue
    # tile grid path: tiles/<grid-x>/<lat-band>/<grid-square>/<y>/<m>/<d>/0/
    gx, lb, gq = scene["tile"][:2], scene["tile"][2], scene["tile"][3:]
    dt = scene["datetime"][:10].split("-")
    prefix = (f"tiles/{gx}/{lb}/{gq}/{dt[0]}/{int(dt[1])}/{int(dt[2])}/0/")
    for band in ("B03", "B04", "B11", "SCL"):
        key = prefix + scene["id"] + ".SAFE/GRANULE/*/IMG_DATA/R20m/"               + f"T{scene['tile']}_{dt[0]}{dt[1]}{dt[2]}T*_B{band}_20m.jp2"
        # list one object under the prefix to get the exact key
        page = s3.list_objects_v2(Bucket="sentinel-s2-l2a",
                                  Prefix=key[:key.index(scene["id"])],
                                  MaxKeys=40)
        hit = next((o["Key"] for o in page.get("Contents", [])
                    if band + "_20m.jp2" in o["Key"]), None)
        if hit is None:
            print(f"  {band}: NOT FOUND in archive")
            continue
        dest = scene_dir / f"{band}.jp2"
        s3.download_file("sentinel-s2-l2a", hit, str(dest))
    print(f"{scene['id']}: downloaded to {scene_dir}")

print("done —", len(catalog['scenes']), "scenes")

